# specset usecase 2

1. Create a Python environment.

2. Install alphapeptdeep, alphabase
    ```
    pip install "alphabase[full-stable]"
    pip install "peptdeep[stable,gui-stable]"
    ```
3. Download pretrained models

    https://github.com/MannLabs/alphapeptdeep/releases/download/pre-trained-models/pretrained_models_v3.zip

4. Install specset
    ```
    pip install specset
    ```

In [1]:
from pathlib import Path

# change path to the data here
p = Path("/d0/tp/scratch/specset_data/hela_ai")

from specset.utils import maxquant_to_specset

print("Create specset from MaxQuant search result...")
maxquant_to_specset(str(p / 'txt'), str(p / 'txt.specset'), additional_columns = {"Score" : "score", "Proteins" : "proteins", "Gene Names" : "genes"})


Create specset from MaxQuant search result...
Reading msms.txt...
Reading evidence.txt...


# To AI data

In [2]:
import os
os.environ["TQDM_DISABLE"] = "1"                  # Suppress tqdm progress bar

import tqdm

from specset.utils import specset_to_alphapeptdeep_string

print("Create alphapeptdeep stream...")
str_stream = specset_to_alphapeptdeep_string(str(p / 'txt.specset'))

dfs = []
frag_inten_dfs = []
from alphabase.spectral_library.reader import LibraryReaderBase
_lib = LibraryReaderBase()

n = 0
for psm in str_stream:    
    n += 1
    if n % 1000 == 0:
        print(f"n = {n}", end = '\r')
    a = _lib.import_file(psm)
    dfs.append(a)
    frag_inten_dfs.append(_lib.fragment_intensity_df)

print(f"Number of PSMs = {n}")

import pickle
f = open(p / 'txt.specset.dfs.pckl', 'wb')
pickle.dump(dfs, f)
f.close()

f = open(p / 'txt.specset.frag_inten_dfs.pckl', 'wb')
pickle.dump(frag_inten_dfs, f)
f.close()

Create alphapeptdeep stream...


/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/alphabase/psm_reader/psm_reader.py:173: FutureWarning: Passed unknown arguments to LibraryReaderBase (fixed_C57=False) will be forbidden in alphabase>1.5.0.
  warnings.warn(


Number of PSMs = 278280


# Refine the generic MS2 model

In [5]:
#pip install "alphabase[full-stable]"
#pip install "peptdeep[stable,gui-stable]"

from pathlib import Path

# change path to the data here
p = Path("/d0/tp/scratch/specset_data/hela_ai")

import torch # noqa: 401, to prevent crash in Mac Arm
import os
import warnings

import pickle

print('Load PSMs...')
with open(p / 'txt.specset.dfs.pckl', 'rb') as fp:
    dfs = pickle.load(fp)
with open(p / 'txt.specset.frag_inten_dfs.pckl', 'rb') as fp:
    frag_inten_dfs = pickle.load(fp)

print('Prepare alphapeptdeep data...')
from alphabase.peptide.fragment import concat_precursor_fragment_dataframes

psm_df, frag_df = concat_precursor_fragment_dataframes(dfs, frag_inten_dfs)

# fine tune
from peptdeep.model.ms2 import pDeepModel

# change path to the alphapeptdeep data here
model_path = p / 'alphapeptdeep_data/generic/ms2.pth'

model_interface = pDeepModel(override_from_weights=True)
model_interface.load(model_path)

supported_fragment_types = model_interface.model.supported_charged_frag_types
print(f"Fragment types supported by the model: {supported_fragment_types}")

psm_df['nce'] = 30.0
psm_df['instrument'] = 'timsTOF'

print('MS2 model adaptation...')

model_interface.train(precursor_df=psm_df, fragment_intensity_df=frag_df.loc[:, supported_fragment_types], 
                      lr= 0.0001,
                      batch_size=512,
                      warmup_epoch=10, epoch=20, verbose=1)

#save model
model_interface.save(str(p / 'transfer_ms2_hela/transfer_ms2.pth'))


#####
print('Generate library using adapted MS2 model...')

from peptdeep.pipeline_api import global_settings, generate_library

fasta = str(p / 'uniprot_Human.fasta')

lib_settings = global_settings['library']

lib_settings['infile_type'] = 'fasta'
lib_settings['infiles'] = [fasta]
lib_settings['fasta']['add_contaminants'] = False
lib_settings['output_tsv']['enabled'] = True
lib_settings['output_folder'] = str(p / 'lib_transfer_ms2_hela')


mgr_settings = global_settings['model_mgr']

mgr_settings['external_ms2_model'] = str(p / 'transfer_ms2_hela/transfer_ms2.pth')
mgr_settings['external_rt_model'] = str(p / 'alphapeptdeep_data/generic/rt.pth')
mgr_settings['external_ccs_model'] = str(p / 'alphapeptdeep_data/generic/ccs.pth')

generate_library()

#####
print('Generate library using the default model...')
lib_settings['output_folder'] = str(p / 'lib_default_hela')

mgr_settings['external_ms2_model'] = str(p / 'alphapeptdeep_data/generic/ms2.pth')
mgr_settings['external_rt_model'] = str(p / 'alphapeptdeep_data/generic/rt.pth')
mgr_settings['external_ccs_model'] = str(p / 'alphapeptdeep_data/generic/ccs.pth')

generate_library()


Load PSMs...
Prepare alphapeptdeep data...
Fragment types supported by the model: ['b_z1', 'b_z2', 'y_z1', 'y_z2', 'b_modloss_z1', 'b_modloss_z2', 'y_modloss_z1', 'y_modloss_z2']
MS2 model adaptation...
2026-10-01 14:15:10> Training with fixed sequence length: 0


/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/peptdeep/model/ms2.py:733: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  to_be_loaded_state_dict = t

[Training] Epoch=1, lr=2e-05, loss=0.01243575816859671
[Training] Epoch=2, lr=3e-05, loss=0.011711144867238768
[Training] Epoch=3, lr=4e-05, loss=0.011576067791892249
[Training] Epoch=4, lr=5e-05, loss=0.011505946532512705
[Training] Epoch=5, lr=6e-05, loss=0.011481784904996553
[Training] Epoch=6, lr=7e-05, loss=0.011399519554617113
[Training] Epoch=7, lr=8e-05, loss=0.011386899187703404
[Training] Epoch=8, lr=9e-05, loss=0.011448843552518571
[Training] Epoch=9, lr=0.0001, loss=0.011339409569719513
[Training] Epoch=10, lr=0.0001, loss=0.011327430539715447
[Training] Epoch=11, lr=9.755282581475769e-05, loss=0.011365287429081243
[Training] Epoch=12, lr=9.045084971874738e-05, loss=0.011322064205121838
[Training] Epoch=13, lr=7.938926261462366e-05, loss=0.011289082726529032
[Training] Epoch=14, lr=6.545084971874738e-05, loss=0.011199506678161
[Training] Epoch=15, lr=5e-05, loss=0.011093435210600626
[Training] Epoch=16, lr=3.4549150281252636e-05, loss=0.011068158552638794
[Training] Epoch=1

/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/alpharaw/raw_access/pysciexwifffilereader.py:45: UserWarning: Dotnet-based dependencies could not be loaded. Sciex support is disabled.
  warnings.warn(
/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/alpharaw/raw_access/pythermorawfilereader.py:36: UserWarning: Dotnet-based dependencies could not be loaded. Thermo support is disabled.
  warnings.warn(
/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/peptdeep/model/ms2.py:416: UserWarning: mask_modloss is deprecated and will be removed in the future. To mask the modloss fragments, the charged_frag_types should not include the modloss fragments.
  warnings.warn(
/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/peptdeep/model/ms2.py:733: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possibl

2026-10-01 14:21:45> Using external ms2 model: '/d0/tp/scratch/specset_data/hela_ai/transfer_ms2_hela/transfer_ms2.pth'


...

2026-10-01 14:54:52> Using external ms2 model: '/d0/tp/scratch/specset_data/hela_ai/alphapeptdeep_data/generic/ms2.pth'


/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/peptdeep/model/model_interface.py:738: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(stre

2026-10-01 14:54:52> Using external rt model: '/d0/tp/scratch/specset_data/hela_ai/alphapeptdeep_data/generic/rt.pth'
2026-10-01 14:54:52> Using external ccs model: '/d0/tp/scratch/specset_data/hela_ai/alphapeptdeep_data/generic/ccs.pth'
2026-10-01 14:54:52> xxx/library.tsv does not exist, use default IRT_PEPTIDE_DF to translate irt
2026-10-01 14:54:52> Generating the spectral library ...


/d0/tp/specset_experiments/venv_peptdeep151/lib/python3.10/site-packages/peptdeep/model/model_interface.py:738: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  torch.load(stre

Predicting charge states for 3447252 peptides ...
2026-10-01 14:58:04> Loaded 4109724 precursors.
2026-10-01 14:58:17> Predicting RT/IM/MS2 for 4109723 precursors ...
2026-10-01 14:58:17> Predicting RT ...
2026-10-01 14:59:19> Predicting mobility ...
2026-10-01 15:01:14> Predicting MS2 ...
2026-10-01 15:03:33> End predicting RT/IM/MS2
2026-10-01 15:03:33> Predicting the spectral library with 4109723 precursors and 287.66M fragments used 18.5086 GB memory
2026-10-01 15:03:33> Saving HDF library to /d0/tp/scratch/specset_data/hela_ai/lib_default_hela/predict.speclib.hdf ...
2026-10-01 15:04:05> Translating to /d0/tp/scratch/specset_data/hela_ai/lib_default_hela/predict.speclib.tsv for DiaNN/Spectronaut...
Translation finished, it will take several minutes to export the rest precursors to the tsv file...
2026-10-01 15:29:27> Library generated!!
